# MFM Guidance Examples: Norway and Smiley

This notebook reruns the two final Meta Flow Map steering examples top to bottom:

1. Uniform square $\rightarrow$ Norway flag colors, using delayed color-only MFM value guidance.
2. Uniform square $\rightarrow$ smiley, using weak position guidance plus delayed color guidance.

The defaults below are the calibrated settings from the experiments. The outputs are saved under `figures/`.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import torch

from data import NUM_CLASSES
from sample import load_model, resolve_figure_path
from value_fn import freeze_model
from mfm_value_fn import load_mfm

from reward import (
    norwegian_color_reward_for_model_state,
    norwegian_color_reward_raw,
)
from experiment_uniform_norway_mfm_guidance import (
    delayed_color_sample_mfm,
    norway_target,
    panel as plot_norway_panel,
    plot_diags as plot_norway_diags,
    summarize as summarize_norway,
)

from dMFM_experiments.reward_smiley import (
    smiley_log_reward_for_model_state,
    smiley_reward_raw,
)
from experiment_uniform_smiley_mfm_guidance import (
    plot_diag as plot_smiley_diags,
    plot_result as plot_smiley_result,
    sample_hybrid_mfm,
    summarize as summarize_smiley,
)

torch.set_grad_enabled(True)
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Shared Model Loading

In [ ]:
BASE_CKPT = "ckpt_uniform_color_square.pt"
MFM_CKPT = "checkpoints/mfm_consistency_10000.pt"

base_model = freeze_model(load_model(BASE_CKPT, device=device, use_ema=True))
mfm = load_mfm(MFM_CKPT, device=device, use_ema=True)

print(f"device={device}")
print(f"base checkpoint: {BASE_CKPT}")
print(f"mfm checkpoint:  {MFM_CKPT}")

## Example 1: Norway

Best current MFM setting for this example:

- `N=10000`
- MFM value estimator with one posterior map step
- MC samples: `8`
- Delayed color guidance window: $t \in [0.8, 0.95]$
- Color guidance ratio: `6.0`
- Reward scale `beta=16`

In [ ]:
NORWAY = {
    "n_samples": 10_000,
    "n_steps": 100,
    "n_value_samples": 8,
    "n_mfm_steps": 1,
    "mfm_final_u": 1.0,
    "beta": 16.0,
    "k": 40.0,
    "color_base_freeze_until": 0.8,
    "guidance_start": 0.8,
    "guidance_end": 0.95,
    "guidance_ratio": 6.0,
    "guidance_batch_size": 1000,
    "base_batch_size": 10_000,
    "seed": 0,
}

def norway_reward_fn(X1):
    return NORWAY["beta"] * norwegian_color_reward_for_model_state(
        X1,
        k=NORWAY["k"],
        color_score="log_softmax",
        detach_regions=True,
    )

In [ ]:
norway_endpoints = {}
norway_diags = {}

for label, ratio in [("unguided", 0.0), ("MFM guided", NORWAY["guidance_ratio"] )]:
    print(f"\n=== Norway: {label} ===")
    t0 = time.time()
    x, diag = delayed_color_sample_mfm(
        base_model,
        mfm,
        n_samples=NORWAY["n_samples"],
        n_steps=NORWAY["n_steps"],
        reward_fn=norway_reward_fn,
        device=device,
        seed=NORWAY["seed"],
        color_base_freeze_until=NORWAY["color_base_freeze_until"],
        guidance_ratio=ratio,
        n_value_samples=NORWAY["n_value_samples"],
        n_mfm_steps=NORWAY["n_mfm_steps"],
        mfm_final_u=NORWAY["mfm_final_u"],
        guidance_start=NORWAY["guidance_start"],
        guidance_end=NORWAY["guidance_end"],
        guidance_batch_size=NORWAY["guidance_batch_size"],
        base_batch_size=NORWAY["base_batch_size"],
        verbose=True,
    )
    print(f"finished in {time.time() - t0:.1f}s")
    summarize_norway(label, x, k=NORWAY["k"])
    norway_endpoints[label] = x.cpu()
    norway_diags[label] = diag

torch.save(
    {"config": NORWAY, "endpoints": norway_endpoints},
    resolve_figure_path("uniform_mfm_notebook_norway_endpoints.pt"),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.6))
for ax, (label, x) in zip(axes, norway_endpoints.items()):
    reward = norwegian_color_reward_raw(
        x,
        k=NORWAY["k"],
        color_score="log_softmax",
        detach_regions=True,
    )
    pred = x[:, :NUM_CLASSES].argmax(dim=-1)
    target = norway_target(x[:, NUM_CLASSES:], k=NORWAY["k"])
    match = (pred == target).float().mean().item()
    plot_norway_panel(
        ax,
        x,
        f"{label}\nreward={reward.mean().item():.3f}  match={match:.3f}",
        marker_size=5.5,
    )
plt.tight_layout()
norway_out = resolve_figure_path("uniform_mfm_notebook_norway.png")
fig.savefig(norway_out, dpi=190)
print(f"saved {norway_out}")
plt.show()

plot_norway_diags(norway_diags, "uniform_mfm_notebook_norway_diagnostics.png")

## Example 2: Smiley

This uses the same larger smiley geometry as the good GLASS run:

- `eye_sigma=0.28`
- `mouth_sigma=0.23`
- `mouth_x_half=0.84`
- `mouth_curve=0.65`

Best current MFM setting for this example:

- `N=1000`
- `200` outer integration steps
- MFM value estimator with one posterior map step
- MC samples: `16`
- Position guidance: `pos_lambda=0.2`, with SDE-sigma scaling
- Delayed color guidance window: $t \in [0.8, 0.98]$
- Color guidance ratio: `2.0`
- Reward scale `beta=64`

In [ ]:
SMILEY = {
    "n_samples": 1000,
    "n_steps": 200,
    "n_value_samples": 16,
    "n_mfm_steps": 1,
    "mfm_final_u": 1.0,
    "beta": 64.0,
    "pos_lambda": 0.2,
    "pos_start": 0.0,
    "pos_end": 0.999,
    "pos_lambda_base": "sde_sigma_sq",
    "color_base_freeze_until": 0.8,
    "color_guidance_ratio": 2.0,
    "color_guidance_start": 0.8,
    "color_guidance_end": 0.98,
    "guidance_batch_size": 500,
    "base_batch_size": 10_000,
    "seed": 0,
}

SMILEY_SHAPE = {
    "eye_sigma": 0.28,
    "mouth_sigma": 0.23,
    "mouth_x_half": 0.84,
    "mouth_y_low": -0.88,
    "mouth_y_high": 0.18,
    "mouth_y0": -0.55,
    "mouth_curve": 0.65,
    "k": 18.0,
    "color_temp": 1.0,
}

def smiley_reward_fn(X1):
    return SMILEY["beta"] * smiley_log_reward_for_model_state(
        X1,
        detach_regions=False,
        **SMILEY_SHAPE,
    )

In [ ]:
smiley_endpoints = {}
smiley_diags = {}

for label, pos_lam, color_ratio in [
    ("baseline", 0.0, 0.0),
    ("MFM guided", SMILEY["pos_lambda"], SMILEY["color_guidance_ratio"]),
]:
    print(f"\n=== Smiley: {label} ===")
    t0 = time.time()
    x, diag = sample_hybrid_mfm(
        base_model,
        mfm,
        n_samples=SMILEY["n_samples"],
        n_steps=SMILEY["n_steps"],
        reward_fn=smiley_reward_fn,
        device=device,
        seed=SMILEY["seed"],
        pos_lambda=pos_lam,
        pos_start=SMILEY["pos_start"],
        pos_end=SMILEY["pos_end"],
        pos_lambda_base=SMILEY["pos_lambda_base"],
        color_base_freeze_until=SMILEY["color_base_freeze_until"],
        color_guidance_ratio=color_ratio,
        color_guidance_start=SMILEY["color_guidance_start"],
        color_guidance_end=SMILEY["color_guidance_end"],
        n_value_samples=SMILEY["n_value_samples"],
        n_mfm_steps=SMILEY["n_mfm_steps"],
        mfm_final_u=SMILEY["mfm_final_u"],
        guidance_batch_size=SMILEY["guidance_batch_size"],
        base_batch_size=SMILEY["base_batch_size"],
        verbose=True,
    )
    print(f"finished in {time.time() - t0:.1f}s")
    summarize_smiley(label, x, SMILEY_SHAPE)
    smiley_endpoints[label] = x.cpu()
    smiley_diags[label] = diag

torch.save(
    {"config": SMILEY, "shape_kwargs": SMILEY_SHAPE, "endpoints": smiley_endpoints},
    resolve_figure_path("uniform_mfm_notebook_smiley_endpoints.pt"),
)

In [ ]:
plot_smiley_result(
    smiley_endpoints,
    SMILEY_SHAPE,
    "uniform_mfm_notebook_smiley.png",
    show_background=True,
)
plt.show()

plot_smiley_result(
    smiley_endpoints,
    SMILEY_SHAPE,
    "uniform_mfm_notebook_smiley_no_background.png",
    show_background=False,
)
plot_smiley_diags(smiley_diags, "uniform_mfm_notebook_smiley_diagnostics.png")